In [1]:
import numpy as np
import pandas as pd

# =====================
# Plotting
# =====================
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
import seaborn as sns

# =====================
# SciPy / Stats
# =====================
from scipy.stats import norm
import statsmodels.api as sm
import statsmodels.formula.api as smf

# =====================
# sklearn – model selection
# =====================
from sklearn.model_selection import train_test_split

# =====================
# sklearn – models
# =====================
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# =====================
# sklearn – metrics
# =====================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    r2_score,
    mean_squared_error,
)

# =====================
# Utilities
# =====================
from itertools import combinations, product
from tqdm import tqdm

# Some auxiliary functions

In [2]:
# Construct the replicator model
def construct_replicator_features(X):
    linear_part = X.values  # shape: [n_samples, 11]

    # Cross-species interactions only (i < j)
    cross_terms = []
    for i, j in combinations(range(X.shape[1]), 2):
        interaction = - (X.iloc[:, i] * X.iloc[:, j]).values.reshape(-1, 1)
        cross_terms.append(interaction)

    quadratic_part = np.hstack(cross_terms)  # shape: [n_samples, 55]
    features = np.hstack([linear_part, quadratic_part])  # shape: [n_samples, 66]
    return features

# Take the mid point of each Nugent score class. Easier to multiply it by 10 here and divide later
def convert_to_levels(y):
    return pd.cut(
        y,
        bins=[-0.1, 3, 6, 10],  # 3 categories
        labels=[15, 50, 85]     # Midpoints
    ).astype(int)

# Nugent Score transformation
def transform_y(y, N0=None, c=None):
    if N0 is None or c is None:
        return y
    N_max = 10
    d = N_max - 2 * N0 + c
    return np.log(N0 + d) - np.log(N_max - y + c)
import matplotlib.pyplot as plt


#Summarize metrics
def summarize_metrics(metrics_dict, name):
    print(f"\n{name} Average Metrics over {N} runs:")
    for key, values in metrics_dict.items():
        print(f"{key.capitalize():<10}: {np.mean(values):.4f} ± {np.std(values):.4f}")

# Importing the data

In [3]:
data = pd.read_csv("10_species_cleaned_with_ethnicity_and_community.csv")

# Extract response and features
y = data.iloc[:, 3].astype(int)
X_raw = data.iloc[:, 4:]

# Convert to frequencies
X_freq = X_raw[sorted(X_raw.columns)]
X_freq = X_freq.iloc[:, 1:] #Remove unassigned
X_freq = X_freq.div(X_freq.sum(axis=1), axis=0)

# Calibrating the Ridge regression parameters (alpha fixed at 0.05)

In [4]:
def run_multiple_evaluations_with_transformed_regression(X, y, N0=None, c=None, N=100, test_size=0.2, random_state_seed=42, alpha=0.05):
    unique_labels = sorted(y.unique())
    metrics_reg = {'accuracy': [], 'precision': [] , 'f1': []}

    for i in range(N):
        X_train, X_test, y_train_raw, y_test_raw = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # Transform y_train and y_test
        y_train_transformed = transform_y(y_train_raw/10, N0, c)
        y_test_transformed = transform_y(y_test_raw/10, N0, c)

        # Replicator features
        X_train_rep = construct_replicator_features(X_train)
        X_test_rep = construct_replicator_features(X_test)

        # Linear regression
        #No intercept
        X_train_sm = X_train_rep
        X_test_sm = X_test_rep

        model = Ridge(alpha=alpha, fit_intercept=False)
        model.fit(X_train_sm, y_train_transformed)


        # Predict and inverse-transform
        y_pred_transformed = model.predict(X_test_sm)

        y_pred_inverse = []
        if N0 is not None and c is not None:
            N_max = y.max()
            #print(N_max)
            for pred in y_pred_transformed:
                pred = max(min(pred, 100), -100)
                N_pred = N_max + c - (N_max - N0 + c) / np.exp(pred)
                closest_label = min(unique_labels, key=lambda x: abs(x - 10 * N_pred))
                y_pred_inverse.append(closest_label)
        else:
            # No transformation, so predict directly and round
            y_pred_inverse = np.round(10*y_pred_transformed).astype(int)
            y_pred_inverse = np.clip(y_pred_inverse, min(unique_labels), max(unique_labels))

        y_pred_inverse = np.array(y_pred_inverse)

        # Metrics
        metrics_reg['accuracy'].append(accuracy_score(y_test_raw, y_pred_inverse))
        metrics_reg['precision'].append(precision_score(y_test_raw, y_pred_inverse, average='weighted', zero_division=0))
        metrics_reg['f1'].append(f1_score(y_test_raw, y_pred_inverse, average='weighted', zero_division=0))

    return metrics_reg

In [5]:
#Repeat the process 100 times to prevent bias
N = 100
results = []

# Parameter sweep
N0_values = np.arange(1, 9, 0.5)
c_values = np.arange(0.5, 5.5, 0.5)


# Apply label conversion for 3 classes
y_levels = convert_to_levels(y)
  
for N0 in N0_values:
    for c in c_values:
        metrics_reg = run_multiple_evaluations_with_transformed_regression(X_freq, y_levels, N0=N0, c=c, N=N)
        results.append({
            'N0': N0,
            'c': c,
            'regression_accuracy': np.mean(metrics_reg['accuracy']),
            'regression_precision': np.mean(metrics_reg['precision']),
            'regression_f1': np.mean(metrics_reg['f1']),
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Display results
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(results_df)

      N0    c  regression_accuracy  regression_precision  regression_f1
0    1.0  0.5             0.241899              0.060459       0.096268
1    1.0  1.0             0.241899              0.060459       0.096268
2    1.0  1.5             0.241899              0.060459       0.096268
3    1.0  2.0             0.241899              0.060459       0.096268
4    1.0  2.5             0.241899              0.060459       0.096268
5    1.0  3.0             0.241899              0.060459       0.096268
6    1.0  3.5             0.241899              0.060459       0.096268
7    1.0  4.0             0.241899              0.060459       0.096268
8    1.0  4.5             0.242405              0.060980       0.096920
9    1.0  5.0             0.242658              0.061324       0.097328
10   1.5  0.5             0.246203              0.069417       0.106766
11   1.5  1.0             0.251266              0.081335       0.120288
12   1.5  1.5             0.256076              0.096303       0

# Ridge regression and fitting

In [6]:
def fit_transformed_ridge_on_full_data(X, y, N0=None, c=None, alpha=1.0, n_boot=1000, random_state=None):
    if random_state:
        np.random.seed(random_state)
    
    # Transform y
    y_transformed = transform_y(y / 10, N0, c)

    # Replicator features
    X_rep = construct_replicator_features(X)

    # Build feature names
    base_names = [
    'Actinomycetota',
    'Bacillota_A_368345',
    'Bacillota_C',
    'Bacillota_I',
    'Bacteroidota',
    'Campylobacterota_A',
    'Fusobacteriota',
    'Patescibacteria',
    'Pseudomonadota',
    'Synergistota'
]
    if X.shape[1] > len(base_names):
        base_names += [f"x{i}" for i in range(len(base_names), X.shape[1])]

    interaction_names = [
        f"{base_names[i]} * {base_names[j]}"
        for i, j in combinations(range(X.shape[1]), 2)
    ]
    feature_names = base_names + interaction_names

    # Fit Ridge regression
    model = Ridge(alpha=alpha, fit_intercept=False)
    model.fit(X_rep, y_transformed)
    r2 = model.score(X_rep, y_transformed)

    # Collect coefficients
    model_params_df = pd.DataFrame([model.coef_], columns=feature_names)


    return model_params_df, model, r2

def fit_transformed_ridge_on_full_data(
    X, y, N0=None, c=None, alpha=1.0, n_boot=1000, random_state=None
):
    if random_state is not None:
        np.random.seed(random_state)

    # Ensure arrays for internal use (but keep original types for nicer indexing if you want)
    X_is_df = isinstance(X, pd.DataFrame)
    y_is_series = isinstance(y, pd.Series)

    # Transform y
    y_transformed = transform_y((y / 10), N0, c)

    # Replicator features (apply to full X)
    X_rep = construct_replicator_features(X)

    # Build feature names
    base_names = [
    'Actinomycetota',
    'Bacillota_A_368345',
    'Bacillota_C',
    'Bacillota_I',
    'Bacteroidota',
    'Campylobacterota_A',
    'Fusobacteriota',
    'Patescibacteria',
    'Pseudomonadota',
    'Synergistota'
    ]
    if X.shape[1] > len(base_names):
        base_names += [f"x{i}" for i in range(len(base_names), X.shape[1])]

    interaction_names = [
        f"{base_names[i]} * {base_names[j]}"
        for i, j in combinations(range(X.shape[1]), 2)
    ]
    feature_names = base_names + interaction_names

    # Fit Ridge on full data
    model = Ridge(alpha=alpha, fit_intercept=False)
    model.fit(X_rep, y_transformed)
    r2 = model.score(X_rep, y_transformed)
    coef_full = model.coef_.copy()

    # Prepare bootstrap storage
    n_samples = X.shape[0]
    p = len(feature_names)
    boot_coefs = np.zeros((n_boot, p))

    # Bootstrap loop (use .iloc for DataFrame row sampling)
    for b in range(n_boot):
        idx = np.random.choice(n_samples, n_samples, replace=True)
        Xb = X.iloc[idx] if X_is_df else X[idx]
        yb = y.iloc[idx] if y_is_series else y[idx]

        yb_transformed = transform_y((yb / 10), N0, c)
        Xb_rep = construct_replicator_features(Xb)

        boot_model = Ridge(alpha=alpha, fit_intercept=False)
        boot_model.fit(Xb_rep, yb_transformed)
        boot_coefs[b, :] = boot_model.coef_

    # Bootstrap summaries
    boot_mean = np.mean(boot_coefs, axis=0)
    boot_std = np.std(boot_coefs, axis=0, ddof=1)

    # 95% percentile confidence intervals
    ci_lower = np.percentile(boot_coefs, 2.5, axis=0)
    ci_upper = np.percentile(boot_coefs, 97.5, axis=0)


    # Assemble results DataFrame
    results_df = pd.DataFrame({
        "coef": coef_full,
        "boot_mean": boot_mean,
        "boot_std": boot_std,
        "ci_lower_95": ci_lower,
        "ci_upper_95": ci_upper
    }, index=feature_names)

    return results_df, model, r2

Getting the coefficients with the full dataset:

In [7]:
y_levels = convert_to_levels(y)

N0 = 4.5
c = 4

results_df, model, r2 = fit_transformed_ridge_on_full_data(
    X_freq, y_levels, N0=N0, c=c, alpha = 0.05
)

print("Estimated Parameters:\n", results_df['coef'].T)

Estimated Parameters:
 Actinomycetota                             0.600138
Bacillota_A_368345                         0.375088
Bacillota_C                                0.617890
Bacillota_I                               -0.231253
Bacteroidota                               0.043400
Campylobacterota_A                        -0.777789
Fusobacteriota                             0.702461
Patescibacteria                            0.528463
Pseudomonadota                             0.025174
Synergistota                              -0.376599
Actinomycetota * Bacillota_A_368345       -0.161388
Actinomycetota * Bacillota_C               0.357053
Actinomycetota * Bacillota_I              -0.628102
Actinomycetota * Bacteroidota             -0.570903
Actinomycetota * Campylobacterota_A        0.042600
Actinomycetota * Fusobacteriota            0.060532
Actinomycetota * Patescibacteria          -0.054010
Actinomycetota * Pseudomonadota            0.089899
Actinomycetota * Synergistota            

# Monte Carlo cross validation Performance results

In [13]:
def run_multiple_evaluations_with_ridge(X, y, N0=None, c=None, N=100, test_size=0.2, alpha = 0.5, random_state_seed=42):

    unique_labels = sorted(y.unique())
    metrics_reg = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    model_params_list = []
    results = []

    for i in range(N):
        # X_train, X_test, y_train_raw, y_test_raw = train_test_split(
        #     X, y, test_size=test_size, random_state=random_state_seed + i
        # )

        # Ensure consistent splits regardless of columns
        train_idx, test_idx = train_test_split(
            X.index, test_size=test_size, random_state=random_state_seed + i
        )
        # if i==0:
        #     print(train_idx)
        
        # Subset using indices
        X_train = X.loc[train_idx]
        X_test = X.loc[test_idx]
        y_train_raw = y.loc[train_idx]
        y_test_raw = y.loc[test_idx]

        # Transform y
        y_train_transformed = transform_y(y_train_raw/10, N0, c)
        y_test_transformed = transform_y(y_test_raw/10, N0, c)

        # Replicator features
        X_train_rep = construct_replicator_features(X_train)
        X_test_rep = construct_replicator_features(X_test)

        # Ridge regression (no intercept)
        if alpha==0:
            model = sm.OLS(y_train_transformed, X_train_rep).fit()
            r20 = model.rsquared
            residuals0 = model.resid
            mse0 = np.mean(residuals0**2)
        else:
            model = Ridge(alpha=alpha, fit_intercept=False)
            model.fit(X_train_rep, y_train_transformed)
            model_params_list.append(model.coef_)

        #print(model.coef_)
        # Predict and inverse-transform
        y_pred_transformed = model.predict(X_test_rep)
        y_pred_inverse = []
        # if i==0:
        #         print(model.coef_)
        
        if N0 is not None and c is not None:
            N_max = 10
            for pred in y_pred_transformed:
                pred = max(min(pred, 100), -100)
                N_pred = N_max + c - (N_max - N0 + c) / np.exp(pred)
                closest_label = min(unique_labels, key=lambda x: abs(x - 10 * N_pred))
                y_pred_inverse.append(closest_label)
        else:
            y_pred_inverse = np.round(10*y_pred_transformed).astype(int)
            y_pred_inverse = np.clip(y_pred_inverse, min(unique_labels), max(unique_labels))

        y_pred_inverse = np.array(y_pred_inverse)

        # Compute metrics
        metrics_reg['accuracy'].append(accuracy_score(y_test_raw, y_pred_inverse))
        metrics_reg['precision'].append(precision_score(y_test_raw, y_pred_inverse, average='weighted', zero_division=0))
        metrics_reg['f1'].append(f1_score(y_test_raw, y_pred_inverse, average='weighted', zero_division=0))

        # Confusion matrix
        cm = confusion_matrix(y_test_raw, y_pred_inverse, labels=unique_labels)
        total_conf_matrix += cm

        # Combine model coefficients into a DataFrame
        n_features = X_train_rep.shape[1]
        param_names = [f'param_{i}' for i in range(n_features)]
        model_params_df = pd.DataFrame(model_params_list, columns=param_names)
    
        r2 = r2_score(y_test_transformed, y_pred_transformed)
        mse = mean_squared_error(y_test_transformed, y_pred_transformed)
    
        # Compute classification metric (accuracy)
        acc = accuracy_score(y_test_raw, y_pred_inverse)

        #print(y_pred_transformed)
        # Store results
        if alpha==0:
            results.append([r20, mse0, acc])
        else:
            results.append([r2, mse, acc])

    return metrics_reg, total_conf_matrix, unique_labels, model_params_df, results

3 classes (0-3, 4-6, 7-10)

In [14]:
# Convert labels
y_levels = convert_to_levels(y)

metrics_reg, conf_matrix_reg, labels_3lvl, model_params_df, results = run_multiple_evaluations_with_ridge(
    X_freq, y_levels, N0=4.5, c=4, N=100, alpha = 0.05
)
N=100;
summarize_metrics(metrics_reg, "Ridge Transformed Replicator Regression")


Ridge Transformed Replicator Regression Average Metrics over 100 runs:
Accuracy  : 0.8286 ± 0.0379
Precision : 0.8187 ± 0.0460
F1        : 0.8200 ± 0.0414


Train on 3 and test on 2 classes (0-6, 7-10)

In [15]:
def run_multiple_evaluations_with_ridge_grouped_eval2(
    X, y, N0=None, c=None, N=100, test_size=0.2, alpha=1.0, random_state_seed=42
):

    unique_labels = sorted(y.unique())
    metrics_reg = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    model_params_list = []
    results = []

    for i in range(N):
        X_train, X_test, y_train_raw, y_test_raw = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # Transform y for regression
        y_train_transformed = transform_y(y_train_raw / 10, N0, c)
        y_test_transformed = transform_y(y_test_raw / 10, N0, c)

        # Replicator features
        X_train_rep = construct_replicator_features(X_train)
        X_test_rep = construct_replicator_features(X_test)

        # Fit Ridge model
        if alpha == 0:
            model = sm.OLS(y_train_transformed, X_train_rep).fit()
            r20 = model.rsquared
            mse0 = np.mean(model.resid ** 2)
        else:
            model = Ridge(alpha=alpha, fit_intercept=False)
            model.fit(X_train_rep, y_train_transformed)
            model_params_list.append(model.coef_)

        # Predict and inverse-transform
        y_pred_transformed = model.predict(X_test_rep)
        y_pred_inverse = []

        if N0 is not None and c is not None:
            N_max = 10
            for pred in y_pred_transformed:
                pred = max(min(pred, 100), -100)
                N_pred = N_max + c - (N_max - N0 + c) / np.exp(pred)
                closest_label = min(unique_labels, key=lambda x: abs(x - 10 * N_pred))
                y_pred_inverse.append(closest_label)
        else:
            y_pred_inverse = np.round(10 * y_pred_transformed).astype(int)
            y_pred_inverse = np.clip(y_pred_inverse, min(unique_labels), max(unique_labels))

        y_pred_inverse = np.array(y_pred_inverse)

        # === Evaluation with 3 levels (for reference) ===
        cm_3lvl = confusion_matrix(y_test_raw, y_pred_inverse, labels=unique_labels)
        total_conf_matrix += cm_3lvl

        # === Convert both true & predicted labels to 2-level groups ===
        def group_to_2levels(y_val):
            mapping = {15: 0, 50: 0, 85: 1}
            #mapping = {10:0, 20:0, 30:0, 40:0, 50:0, 60:0, 70:1, 80:1, 90:1, 100:1}
            return np.array([mapping[v] for v in y_val])

        y_test_grouped = group_to_2levels(y_test_raw)
        y_pred_grouped = group_to_2levels(y_pred_inverse)

        # === Compute metrics on grouped (2-level) labels ===
        acc = accuracy_score(y_test_grouped, y_pred_grouped)
        prec = precision_score(y_test_grouped, y_pred_grouped, average='weighted', zero_division=0)
        f1 = f1_score(y_test_grouped, y_pred_grouped, average='weighted', zero_division=0)

        
        metrics_reg['accuracy'].append(acc)
        metrics_reg['precision'].append(prec)
        metrics_reg['f1'].append(f1)
        
        # Regression metrics
        r2 = r2_score(y_test_transformed, y_pred_transformed)
        mse = mean_squared_error(y_test_transformed, y_pred_transformed)

        if alpha == 0:
            results.append([r20, mse0, acc])
        else:
            results.append([r2, mse, acc])

    # Combine model coefficients into a DataFrame
    if model_params_list:
        n_features = X_train_rep.shape[1]
        param_names = [f'param_{i}' for i in range(n_features)]
        model_params_df = pd.DataFrame(model_params_list, columns=param_names)
    else:
        model_params_df = pd.DataFrame()

    return metrics_reg, total_conf_matrix, unique_labels, model_params_df, results


In [16]:
metrics_reg, conf_matrix_reg, labels_3lvl, model_params_df, results = run_multiple_evaluations_with_ridge_grouped_eval2(
    X_freq, y_levels, N0=4.5, c=4, N=100, alpha = 0.05
)

summarize_metrics(metrics_reg, "Ridge Transformed Replicator Regression (Grouped 2-level Eval)")


Ridge Transformed Replicator Regression (Grouped 2-level Eval) Average Metrics over 100 runs:
Accuracy  : 0.9257 ± 0.0259
Precision : 0.9279 ± 0.0250
F1        : 0.9258 ± 0.0258


# Performance by ethnicity (requires that we use the dataset differently)

In [17]:
# Extract the ethnicity column
meta_col = data.iloc[:, 2]  

# Extract Nugent Score (at index 3)
y = data.iloc[:, 3].astype(int)

# Features start from column index 4 onward
X_raw = data.iloc[:, 4:]

# Convert to frequencies per row
X_freq = X_raw.div(X_raw.sum(axis=1), axis=0)

X_freq = X_freq[sorted(X_freq.columns)].iloc[:,1:]
X_freq.insert(0, meta_col.name, meta_col)

In [18]:
def run_multiple_evaluations_with_metrics_ridge_by_race(
    X, y, N0=None, c=None, alpha=1.0, N=100, test_size=0.2, random_state_seed=42
):
    """
    Evaluate Ridge Regression (with transformation) and Random Forest
    over multiple random splits and across race groups.
    """

    # Extract race and features
    race_column = X.iloc[:, 0]
    X_features = X.iloc[:, 1:]
    unique_labels = sorted(y.unique())
    unique_races = sorted(race_column.unique())
    num_classes = len(unique_labels)
    all_groups = unique_races + ['overall']

    # Initialize confusion matrices and metrics
    conf_matrices_ridge = {g: np.zeros((num_classes, num_classes), dtype=int) for g in all_groups}
    conf_matrices_rf = {g: np.zeros((num_classes, num_classes), dtype=int) for g in all_groups}
    metrics_ridge = {g: {'accuracy': [], 'precision': [], 'f1': []} for g in all_groups}
    metrics_rf = {g: {'accuracy': [], 'precision': [], 'f1': []} for g in all_groups}

    model_coefs_list = []  # store ridge coefficients for inspection

    for i in range(N):
        train_idx, test_idx = train_test_split(
            X.index, test_size=test_size, random_state=random_state_seed + i
        )
        
        # Subset using indices
        X_train_full = X.loc[train_idx]
        X_test_full = X.loc[test_idx]
        y_train = y.loc[train_idx]
        y_test = y.loc[test_idx]

        race_train = X_train_full.iloc[:, 0]
        race_test = X_test_full.iloc[:, 0]
        X_train = X_train_full.iloc[:, 1:]
        X_test = X_test_full.iloc[:, 1:]

        # Transform response
        y_train_transformed = transform_y(y_train / 10, N0, c)
        y_test_transformed = transform_y(y_test / 10, N0, c)

        # Replicator features
        X_train_rep = construct_replicator_features(X_train)
        X_test_rep = construct_replicator_features(X_test)

        # Fit Ridge regression
        ridge_model = Ridge(alpha=alpha, fit_intercept=False)
        ridge_model.fit(X_train_rep, y_train_transformed)
        model_coefs_list.append(ridge_model.coef_)
        
        # Predict and inverse-transform
        y_pred_transformed = ridge_model.predict(X_test_rep)

        y_pred_inverse = []
        N_max = 10
        for pred in y_pred_transformed:
            pred = np.clip(pred, -100, 100)
            N_pred = N_max + c - (N_max - N0 + c) / np.exp(pred)
            closest_label = min(unique_labels, key=lambda x: abs(x - 10 * N_pred))
            y_pred_inverse.append(closest_label)
        y_pred_inverse = np.array(y_pred_inverse)

        # Random Forest baseline
        rf = RandomForestClassifier(n_estimators=200, random_state=random_state_seed + i)
        rf.fit(X_train, y_train)
        y_pred_rf = rf.predict(X_test)

        # Compute metrics overall
        cm_ridge = confusion_matrix(y_test, y_pred_inverse, labels=unique_labels)
        cm_rf = confusion_matrix(y_test, y_pred_rf, labels=unique_labels)
        conf_matrices_ridge['overall'] += cm_ridge
        conf_matrices_rf['overall'] += cm_rf

        for metrics_dict, y_pred in [
            (metrics_ridge['overall'], y_pred_inverse),
            (metrics_rf['overall'], y_pred_rf),
        ]:
            metrics_dict['accuracy'].append(accuracy_score(y_test, y_pred))
            metrics_dict['precision'].append(precision_score(y_test, y_pred, average='weighted', zero_division=0))
            metrics_dict['f1'].append(f1_score(y_test, y_pred, average='weighted', zero_division=0))

        # Per-race metrics
        for race in unique_races:
            race_mask = race_test == race
            if not race_mask.any():
                continue

            y_test_race = y_test[race_mask]
            y_pred_ridge_race = y_pred_inverse[race_mask]
            y_pred_rf_race = y_pred_rf[race_mask]

            cm_ridge_race = confusion_matrix(y_test_race, y_pred_ridge_race, labels=unique_labels)
            cm_rf_race = confusion_matrix(y_test_race, y_pred_rf_race, labels=unique_labels)
            conf_matrices_ridge[race] += cm_ridge_race
            conf_matrices_rf[race] += cm_rf_race

            for metrics_dict, y_true, y_pred in [
                (metrics_ridge[race], y_test_race, y_pred_ridge_race),
                (metrics_rf[race], y_test_race, y_pred_rf_race),
            ]:
                metrics_dict['accuracy'].append(accuracy_score(y_true, y_pred))
                metrics_dict['precision'].append(precision_score(y_true, y_pred, average='weighted', zero_division=0))
                metrics_dict['f1'].append(f1_score(y_true, y_pred, average='weighted', zero_division=0))

    # Average metrics
    def summarize_metrics(metrics_dict):
        summary = {}
        for group in all_groups:
            summary[group] = {}
            for m in ['accuracy', 'precision', 'f1']:
                vals = metrics_dict[group][m]
                if vals:
                    summary[group][m] = {'mean': np.mean(vals), 'std': np.std(vals), 'n': len(vals)}
                else:
                    summary[group][m] = {'mean': np.nan, 'std': np.nan, 'n': 0}
        return summary

    stats_ridge = summarize_metrics(metrics_ridge)
    stats_rf = summarize_metrics(metrics_rf)

    # Collect model coefficients (overall)
    feature_names = X_train_rep.columns if hasattr(X_train_rep, 'columns') else [f"feature_{i}" for i in range(X_train_rep.shape[1])]
    model_coefs_df = pd.DataFrame(model_coefs_list, columns=feature_names)

    return (
        conf_matrices_ridge, conf_matrices_rf, unique_labels, unique_races,
        stats_ridge, stats_rf, model_coefs_df
    )




def display_race_performance_results(stats_reg, stats_rf, unique_races):
    """Display performance results in the requested format"""
    
    # Display group-wise performance for Replicator Regression
    print("Replicator Regression - Group-wise Performance (mean ± std):")
    for race in unique_races:
        acc = stats_reg[race]['accuracy']
        prec = stats_reg[race]['precision']
        f1 = stats_reg[race]['f1']
        
        if not np.isnan(acc['mean']):
            print(f"Group {race}: Accuracy={acc['mean']:.3f}±{acc['std']:.3f}, "
                  f"Precision={prec['mean']:.3f}±{prec['std']:.3f}, "
                  f"F1={f1['mean']:.3f}±{f1['std']:.3f}")
        else:
            print(f"Group {race}: No data available")
    
    print()
    
    # Display group-wise performance for Random Forest
    print("Random Forest - Group-wise Performance (mean ± std):")
    for race in unique_races:
        acc = stats_rf[race]['accuracy']
        prec = stats_rf[race]['precision']
        f1 = stats_rf[race]['f1']
        
        if not np.isnan(acc['mean']):
            print(f"Group {race}: Accuracy={acc['mean']:.3f}±{acc['std']:.3f}, "
                  f"Precision={prec['mean']:.3f}±{prec['std']:.3f}, "
                  f"F1={f1['mean']:.3f}±{f1['std']:.3f}")
        else:
            print(f"Group {race}: No data available")
    
    print()
    
    # Display overall performance for Regression
    print("Regression Overall Average Metrics over 100 runs:")
    overall_reg = stats_reg['overall']
    print(f"Accuracy  : {overall_reg['accuracy']['mean']:.4f} ± {overall_reg['accuracy']['std']:.4f}")
    print(f"Precision : {overall_reg['precision']['mean']:.4f} ± {overall_reg['precision']['std']:.4f}")
    print(f"F1        : {overall_reg['f1']['mean']:.4f} ± {overall_reg['f1']['std']:.4f}")
    
    print()
    
    # Display overall performance for Random Forest
    print("Random Forest Overall Average Metrics over 100 runs:")
    overall_rf = stats_rf['overall']
    print(f"Accuracy  : {overall_rf['accuracy']['mean']:.4f} ± {overall_rf['accuracy']['std']:.4f}")
    print(f"Precision : {overall_rf['precision']['mean']:.4f} ± {overall_rf['precision']['std']:.4f}")
    print(f"F1        : {overall_rf['f1']['mean']:.4f} ± {overall_rf['f1']['std']:.4f}")


100 Monte Carlo cross validation with 80/20 train/test split

In [20]:
def convert_to_levels2(y):
    return pd.cut(
        y,
        bins=[-0.1, 6, 10],  # 2 categories
        labels=[30, 85]     # Midpoints
    ).astype(int)

N = 100
N0 = 4.5
c = 4
y_levels = convert_to_levels2(y)


conf_matrices_ridge, conf_matrices_rf, unique_labels, unique_races, stats_ridge, stats_rf, model_coefs_df = run_multiple_evaluations_with_metrics_ridge_by_race(
    X_freq, 
    y_levels, 
    N0=N0, 
    c=c, 
    alpha = 0.05,     
    N=N, 
    test_size=0.2
)

display_race_performance_results(stats_ridge, stats_rf, unique_races)


Replicator Regression - Group-wise Performance (mean ± std):
Group Asian: Accuracy=0.918±0.054, Precision=0.939±0.052, F1=0.924±0.053
Group Black: Accuracy=0.934±0.050, Precision=0.938±0.049, F1=0.933±0.050
Group Hispanic: Accuracy=0.891±0.064, Precision=0.904±0.059, F1=0.892±0.063
Group White: Accuracy=0.954±0.043, Precision=0.974±0.031, F1=0.961±0.037

Random Forest - Group-wise Performance (mean ± std):
Group Asian: Accuracy=0.916±0.059, Precision=0.929±0.062, F1=0.917±0.063
Group Black: Accuracy=0.934±0.055, Precision=0.938±0.053, F1=0.934±0.056
Group Hispanic: Accuracy=0.887±0.064, Precision=0.897±0.062, F1=0.887±0.064
Group White: Accuracy=0.972±0.033, Precision=0.980±0.030, F1=0.974±0.032

Regression Overall Average Metrics over 100 runs:
Accuracy  : 0.9237 ± 0.0268
Precision : 0.9281 ± 0.0250
F1        : 0.9248 ± 0.0262

Random Forest Overall Average Metrics over 100 runs:
Accuracy  : 0.9266 ± 0.0258
Precision : 0.9287 ± 0.0254
F1        : 0.9267 ± 0.0258


# Machine Learning algorithms

## Random Forest

In [21]:
def run_multiple_evaluations_with_rf_grouped_eval2(
    X, y, N=100, test_size=0.2, random_state_seed=42
):

    unique_labels = sorted(y.unique())
    metrics_rf = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    feature_importances_list = []
    results = []

    for i in range(N):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # === Fit Random Forest ===
        rf = RandomForestClassifier(n_estimators=100, random_state=random_state_seed + i)
        rf.fit(X_train, y_train)
        y_pred_rf = rf.predict(X_test)

        # === Store feature importances ===
        feature_importances_list.append(rf.feature_importances_)


        # === Compute metrics on grouped (2-level) labels ===
        acc = accuracy_score(y_test, y_pred_rf)
        prec = precision_score(y_test, y_pred_rf, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)

        metrics_rf['accuracy'].append(acc)
        metrics_rf['precision'].append(prec)
        metrics_rf['f1'].append(f1)

        # Save metrics per run
        results.append([acc, prec, f1])

    # Combine feature importances into a DataFrame
    if feature_importances_list:
        n_features = X.shape[1]
        feature_names = getattr(X, 'columns', [f'feature_{i}' for i in range(n_features)])
        feature_importances_df = pd.DataFrame(feature_importances_list, columns=feature_names)
    else:
        feature_importances_df = pd.DataFrame()

    return metrics_rf, feature_importances_df, results


3 Nugent score levels

In [22]:
y_levels = convert_to_levels(y)

metrics_reg, feature_importances_df, results = run_multiple_evaluations_with_rf_grouped_eval2(
    X_freq.iloc[:,1:], y_levels, N=100
)

summarize_metrics(metrics_reg, "Random Forest(Grouped 2-level Eval)")


Random Forest(Grouped 2-level Eval) Average Metrics over 100 runs:
Accuracy  : 0.8118 ± 0.0386
Precision : 0.7913 ± 0.0478
F1        : 0.7937 ± 0.0423


2 Nugent score levels

In [23]:
y_levels = convert_to_levels2(y)

metrics_reg, feature_importances_df, results = run_multiple_evaluations_with_rf_grouped_eval2(
    X_freq.iloc[:,1:], y_levels, N=100
)

summarize_metrics(metrics_reg, "Random Forest(Grouped 2-level Eval)")


Random Forest(Grouped 2-level Eval) Average Metrics over 100 runs:
Accuracy  : 0.9252 ± 0.0250
Precision : 0.9274 ± 0.0248
F1        : 0.9253 ± 0.0250


## XGBoost

In [24]:
def run_multiple_evaluations_with_xgb_grouped_eval2(
    X, y, N=100, test_size=0.2, random_state_seed=42
):

    unique_labels = sorted(y.unique())
    metrics_xgb = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    results = []

    for i in range(N):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # === Fit XGBoost ===
        xgb = XGBClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='mlogloss',
            random_state=random_state_seed + i,
            n_jobs=-1
        )
        xgb.fit(X_train, y_train)
        y_pred_xgb = xgb.predict(X_test)

        acc = accuracy_score(y_test, y_pred_xgb)
        prec = precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0)

        metrics_xgb['accuracy'].append(acc)
        metrics_xgb['precision'].append(prec)
        metrics_xgb['f1'].append(f1)

        results.append([acc, prec, f1])

    return metrics_xgb, total_conf_matrix, results

In [25]:
#Requires binary classification
def convert_to_levels3(y):
    return pd.cut(
        y,
        bins=[-0.1, 6, 10],  # 3 categories
        labels=[0, 1]     # Midpoints
    ).astype(int)



y_levels = convert_to_levels3(y)

metrics_reg, conf_matrix_reg, results = run_multiple_evaluations_with_xgb_grouped_eval2(
    X_freq.iloc[:,1:], y_levels, N=100
)

summarize_metrics(metrics_reg, "XGBOOST 2 LEVELS")


XGBOOST 2 LEVELS Average Metrics over 100 runs:
Accuracy  : 0.9146 ± 0.0276
Precision : 0.9178 ± 0.0272
F1        : 0.9151 ± 0.0274


## SVM

In [26]:
def run_multiple_evaluations_with_svm_grouped_eval2(
    X, y, N=100, test_size=0.2, random_state_seed=42
):
    unique_labels = sorted(y.unique())
    metrics_svm = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    results = []

    for i in range(N):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # === Fit SVM ===
        svm = SVC(
            kernel='rbf',        # you can change to 'linear', 'poly', etc.
            C=1.0,
            gamma='scale',
            random_state=random_state_seed + i
        )
        svm.fit(X_train, y_train)
        y_pred_svm = svm.predict(X_test)

        # === Compute metrics ===
        acc = accuracy_score(y_test, y_pred_svm)
        prec = precision_score(y_test, y_pred_svm, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred_svm, average='weighted', zero_division=0)

        metrics_svm['accuracy'].append(acc)
        metrics_svm['precision'].append(prec)
        metrics_svm['f1'].append(f1)

        results.append([acc, prec, f1])

        # Optionally accumulate confusion matrix
        cm = confusion_matrix(y_test, y_pred_svm, labels=unique_labels)
        total_conf_matrix += cm

    # No feature importances for SVM
    feature_importances_df = pd.DataFrame()

    return metrics_svm, total_conf_matrix, unique_labels, feature_importances_df, results


In [27]:
metrics_reg, conf_matrix_reg, labels_3lvl, model_params_df, results = run_multiple_evaluations_with_svm_grouped_eval2(
    X_freq.iloc[:, 1:], y_levels, N=100
)

summarize_metrics(metrics_reg, "SVM 2 LEVELS")


SVM 2 LEVELS Average Metrics over 100 runs:
Accuracy  : 0.9241 ± 0.0261
Precision : 0.9270 ± 0.0246
F1        : 0.9245 ± 0.0256


## KNN

In [28]:
def run_multiple_evaluations_with_knn_grouped_eval2(
    X, y, N=100, test_size=0.2, random_state_seed=42, n_neighbors=5
):
    unique_labels = sorted(y.unique())
    metrics_knn = {'accuracy': [], 'precision': [], 'f1': []}
    total_conf_matrix = np.zeros((len(unique_labels), len(unique_labels)), dtype=int)
    results = []

    for i in range(N):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state_seed + i
        )

        # === Fit KNN ===
        knn = KNeighborsClassifier(n_neighbors=n_neighbors)
        knn.fit(X_train, y_train)
        y_pred_knn = knn.predict(X_test)

        # === Compute metrics ===
        acc = accuracy_score(y_test, y_pred_knn)
        prec = precision_score(y_test, y_pred_knn, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred_knn, average='weighted', zero_division=0)

        metrics_knn['accuracy'].append(acc)
        metrics_knn['precision'].append(prec)
        metrics_knn['f1'].append(f1)

        results.append([acc, prec, f1])

        # Optionally accumulate confusion matrix
        cm = confusion_matrix(y_test, y_pred_knn, labels=unique_labels)
        total_conf_matrix += cm


    return metrics_knn, total_conf_matrix, results

In [29]:
metrics_reg, conf_matrix_reg, results = run_multiple_evaluations_with_knn_grouped_eval2(
    X_freq.iloc[:,1:], y_levels, N=100
)

summarize_metrics(metrics_reg, "SVM 2 LEVELS")


SVM 2 LEVELS Average Metrics over 100 runs:
Accuracy  : 0.9292 ± 0.0270
Precision : 0.9323 ± 0.0252
F1        : 0.9298 ± 0.0264
